# 自選股數據與多因子選股（第三節課作業）

標的：Palantir（PLTR，美股，Moomoo API）、IREN（美股，Moomoo API）、
BHP（澳股 BHP.AX，yfinance）

> ⚠️ 免責聲明：本 Notebook 之數據與分析僅供教育與研究用途，不構成任何投資建議。

**前置需求**：PLTR / IREN 需要 Moomoo OpenD 已在本機啟動並登入；BHP 使用 yfinance，不需要任何帳戶。

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

from loaders import moomoo_loader, yfinance_loader, unify_schema

plt.rcParams["figure.figsize"] = (11, 4)
print("套件載入完成")

## 1. Palantir（PLTR）— Moomoo API

In [ ]:
print("連線檢查：", moomoo_loader.check_connection())
print("額度狀況：", moomoo_loader.check_quota())

In [ ]:
pltr_daily = moomoo_loader.get_daily_kline("US.PLTR", "2025-01-01", "2025-12-31")
pltr_daily = unify_schema.unify_kline(pltr_daily, "PLTR", "daily")
pltr_daily.tail()

In [ ]:
fig, ax = plt.subplots()
ax.plot(pltr_daily["date"], pltr_daily["close"], color="#1B2CC1")
ax.set_title("PLTR 收盤價（日線, USD）")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.show()

In [ ]:
pltr_financials = moomoo_loader.get_financials("US.PLTR", num_quarters=8)
# 第一次執行請務必檢查實際欄位名稱：
print(pltr_financials.columns.tolist())
pltr_financials

## 2. IREN — Moomoo API

註：IREN 雖然總部在雪梨，但股票僅在美股 Nasdaq 掛牌，並無 ASX 上市，
因此與 PLTR 採同一套資料源（Moomoo API），而非 yfinance。

In [ ]:
iren_daily = moomoo_loader.get_daily_kline("US.IREN", "2025-01-01", "2025-12-31")
iren_daily = unify_schema.unify_kline(iren_daily, "IREN", "daily")

fig, ax = plt.subplots()
ax.plot(iren_daily["date"], iren_daily["close"], color="#7692FF")
ax.set_title("IREN 收盤價（日線, USD）")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.show()

In [ ]:
iren_financials = moomoo_loader.get_financials("US.IREN", num_quarters=8)
iren_financials

## 3. BHP（必和必拓）— yfinance（BHP.AX，AUD 計價）

註：BHP 為雙重掛牌（ASX + NYSE ADR），此處抓取的是 ASX 版本（AUD），
與 PLTR / IREN 的 USD 計價不可直接放在同一張圖上比較。

In [ ]:
bhp_daily = yfinance_loader.get_daily_kline("BHP.AX", "2025-01-01", "2025-12-31")
bhp_daily = unify_schema.unify_kline(bhp_daily, "BHP", "daily")

fig, ax = plt.subplots()
ax.plot(bhp_daily["date"], bhp_daily["close"], color="#ABD2FA")
ax.set_title("BHP.AX 收盤價（日線, AUD）")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.show()

In [ ]:
bhp_financials = yfinance_loader.get_financials("BHP.AX", num_quarters=8)
bhp_financials

## 下一步：多因子選股（Multi-factor Stock Selection）

以上僅完成作業的第一部分（資料獲取）。下一步可以：

1. 從財務數據中挑選幾個因子（例如 ROE、營收成長率、毛利率）
2. 對三檔標的分別計算因子分數，並排序比較
3. **注意前視偏差（look-ahead bias）**：務必使用財報「公告日期」而非
   「報告期間」來對齊資料，避免用到當時還未公布的數字
4. 這部分邏輯建議另外開一個 Notebook 區塊或新檔案 `factor_scoring.py`
   撰寫，維持模組化